# Anomaly Detection Models - Evaluation Metrics Comparison
**Clean Implementation | Precision, Recall, F1-Score | Hackathon Ready**

Evaluate 3 anomaly detection models with comprehensive metrics:
- **Z-Score**: Statistical univariate method
- **Isolation Forest**: Tree-based ensemble method
- **LSTM Autoencoder**: Deep learning reconstruction method

All models evaluated with proper precision, recall, and F1-score metrics from `sklearn.metrics`

## 1. Setup & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.ensemble import IsolationForest
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ All imports successful')
print(f'✓ TensorFlow version: {tf.__version__}')

## 2. Load & Preprocess Data

In [ ]:
# Load data
csv_path = 'Production System Dataset.csv'
df = pd.read_csv(csv_path)

# Clean: keep numeric columns, remove NaN and duplicates
df_numeric = df.select_dtypes(include=[np.number])
df_clean = df_numeric.dropna().drop_duplicates().reset_index(drop=True)

print(f'✓ Data loaded: {df_clean.shape}')
print(f'  Samples: {len(df_clean)} | Features: {len(df_clean.columns)}')
print(f'\n  Sample statistics:')
print(df_clean.describe().iloc[:, :3])

# Scale data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_clean)
print(f'\n✓ Data scaled using StandardScaler')

## 3. Create Ground Truth Labels (Statistical Methods)

In [ ]:
def create_ground_truth(data_scaled, method='zscore_ensemble', contamination=0.07):
    """
    Create ground truth labels using statistical methods.
    
    Args:
        data_scaled: Scaled feature matrix
        method: 'zscore_ensemble' | 'isolation_forest' | 'hybrid'
        contamination: Expected anomaly rate (0-1)
    
    Returns:
        labels: Binary array (0=normal, 1=anomaly)
    """
    
    if method == 'zscore_ensemble':
        # Z-Score: Flag if any feature > 3 std from mean
        z_scores = np.abs(data_scaled)
        composite_z = np.max(z_scores, axis=1)
        threshold = np.percentile(composite_z, 100 * (1 - contamination))
        labels_z = (composite_z > threshold).astype(int)
        return labels_z, threshold, 'Z-Score (Composite max)'
    
    elif method == 'isolation_forest':
        # Isolation Forest
        iso = IsolationForest(contamination=contamination, random_state=42)
        labels = iso.fit_predict(data_scaled)
        return (labels == -1).astype(int), contamination, 'Isolation Forest'
    
    elif method == 'hybrid':
        # Combine Z-Score and Isolation Forest (ensemble)
        z_scores = np.abs(data_scaled)
        composite_z = np.max(z_scores, axis=1)
        threshold_z = np.percentile(composite_z, 100 * (1 - contamination))
        labels_z = (composite_z > threshold_z).astype(int)
        
        iso = IsolationForest(contamination=contamination, random_state=42)
        labels_iso = (iso.fit_predict(data_scaled) == -1).astype(int)
        
        # Union: anomaly if either method agrees
        labels_hybrid = np.maximum(labels_z, labels_iso)
        return labels_hybrid, (threshold_z, contamination), 'Z-Score + Isolation Forest (Hybrid)'

# Create ground truth using hybrid method for best coverage
ground_truth, thresholds, method_name = create_ground_truth(data_scaled, method='hybrid', contamination=0.07)

print(f'✓ Ground Truth Labels Created: {method_name}')
print(f'  Normal:  {np.sum(ground_truth == 0):,} ({100*np.sum(ground_truth == 0)/len(ground_truth):.1f}%)')
print(f'  Anomaly: {np.sum(ground_truth == 1):,} ({100*np.sum(ground_truth == 1)/len(ground_truth):.1f}%)')

## 4. Model 1: Z-Score Anomaly Detection

In [ ]:
class ZScoreDetector:
    """Z-Score based anomaly detector."""
    
    def __init__(self, threshold=3.0):
        self.threshold = threshold
        self.mean = None
        self.std = None
    
    def fit(self, X):
        """Fit on training data (already scaled)."""
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
    
    def predict(self, X, threshold=None):
        """Predict anomalies: 0=normal, 1=anomaly."""
        if threshold is None:
            threshold = self.threshold
        
        z_scores = np.abs(X)
        # Max Z-score across all features
        anomaly_scores = np.max(z_scores, axis=1)
        return (anomaly_scores > threshold).astype(int), anomaly_scores

# Train Z-Score detector
zscore_detector = ZScoreDetector(threshold=3.0)
zscore_detector.fit(data_scaled)

# Get optimal threshold using ROC curve on ground truth
z_scores_abs = np.abs(data_scaled)
anomaly_scores_z = np.max(z_scores_abs, axis=1)
fpr, tpr, thresholds = roc_curve(ground_truth, anomaly_scores_z)
roc_auc = auc(fpr, tpr)

# Find optimal threshold
youden_index = tpr - fpr
optimal_idx = np.argmax(youden_index)
optimal_threshold_z = thresholds[optimal_idx]

predictions_z, scores_z = zscore_detector.predict(data_scaled, threshold=optimal_threshold_z)

print(f'✓ Z-Score Model')
print(f'  Optimal threshold (ROC-AUC): {optimal_threshold_z:.4f}')
print(f'  ROC-AUC Score: {roc_auc:.4f}')
print(f'  Anomalies detected: {np.sum(predictions_z)}')

## 5. Model 2: Isolation Forest

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.07, random_state=42)
predictions_iso = iso_forest.fit_predict(data_scaled)
predictions_iso = (predictions_iso == -1).astype(int)

# Get anomaly scores for ROC curve
anomaly_scores_iso = iso_forest.score_samples(data_scaled)

print(f'✓ Isolation Forest Model')
print(f'  Contamination: 0.07 (7%)')
print(f'  Anomalies detected: {np.sum(predictions_iso)}')

## 6. Model 3: LSTM Autoencoder

In [ ]:
def create_sequences(data, seq_len):
    """Create sliding window sequences."""
    X = []
    for i in range(len(data) - seq_len + 1):
        X.append(data[i:i + seq_len])
    return np.array(X)

# Create sequences
seq_len = 30
X_sequences = create_sequences(data_scaled, seq_len)
y_sequences = ground_truth[seq_len-1:]  # Labels for sequence end

print(f'✓ Sequences created: {X_sequences.shape}')
print(f'  Sequence length: {seq_len}')
print(f'  Total sequences: {len(X_sequences)}')

# Split data: 70% train (NORMAL ONLY), 30% test
train_size = int(0.70 * len(X_sequences))
X_train = X_sequences[:train_size]
y_train = y_sequences[:train_size]
X_test_seq = X_sequences[train_size:]
y_test_seq = y_sequences[train_size:]

# IMPORTANT: Train only on normal data
normal_idx = np.where(y_train == 0)[0]
X_train_normal = X_train[normal_idx]

print(f'\n  Train normal: {len(X_train_normal)} (anomalies excluded: {len(X_train) - len(X_train_normal)})')
print(f'  Test: {len(X_test_seq)}')

In [ ]:
# Build LSTM Autoencoder
n_features = data_scaled.shape[1]

inputs = Input(shape=(seq_len, n_features))

# Encoder
encoded = LSTM(64, activation='relu', return_sequences=True)(inputs)
encoded = Dropout(0.2)(encoded)
encoded = LSTM(32, activation='relu', return_sequences=False)(encoded)

# Decoder
decoded = RepeatVector(seq_len)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(decoded)
decoded = Dropout(0.2)(decoded)
decoded = LSTM(64, activation='relu', return_sequences=True)(decoded)
decoded = TimeDistributed(Dense(n_features))(decoded)

lstm_model = Model(inputs, decoded)
lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print(f'✓ LSTM Autoencoder built')
print(f'  Input shape: (batch, {seq_len}, {n_features})')
print(f'  Parameters: {lstm_model.count_params():,}')

In [ ]:
# Train on normal data only
print('Training LSTM on normal data...')

history = lstm_model.fit(
    X_train_normal, X_train_normal,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print(f'✓ LSTM training complete')
print(f'  Epochs: {len(history.history["loss"])}')
print(f'  Final loss: {history.history["loss"][-1]:.6f}')

In [ ]:
# Predict and compute reconstruction errors
predictions_lstm_full = lstm_model.predict(X_sequences, verbose=0)
mse_errors = np.mean(np.power(X_sequences - predictions_lstm_full, 2), axis=(1, 2))

# Find optimal LSTM threshold
fpr_lstm, tpr_lstm, thresholds_lstm = roc_curve(y_sequences, mse_errors)
roc_auc_lstm = auc(fpr_lstm, tpr_lstm)
youden_lstm = tpr_lstm - fpr_lstm
optimal_idx_lstm = np.argmax(youden_lstm)
optimal_threshold_lstm = thresholds_lstm[optimal_idx_lstm]

predictions_lstm = (mse_errors > optimal_threshold_lstm).astype(int)

print(f'✓ LSTM inference complete')
print(f'  MSE range: [{mse_errors.min():.4f}, {mse_errors.max():.4f}]')
print(f'  Optimal threshold: {optimal_threshold_lstm:.4f}')
print(f'  ROC-AUC: {roc_auc_lstm:.4f}')
print(f'  Anomalies detected: {np.sum(predictions_lstm)}')

## 7. Compute Evaluation Metrics

In [ ]:
def safe_metrics(y_true, y_pred, model_name):
    """
    Safely compute metrics with edge case handling.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        model_name: Name of the model
    
    Returns:
        dict: Metrics dictionary
    """
    
    # Edge case: no predictions or no ground truth
    if len(np.unique(y_pred)) == 1 and np.unique(y_pred)[0] == 0:
        # Model predicts only normal
        precision = 1.0 if np.sum(y_true) == 0 else 0.0
        recall = 0.0
        f1 = 0.0
    elif len(np.unique(y_pred)) == 1 and np.unique(y_pred)[0] == 1:
        # Model predicts everything as anomaly
        precision = np.sum(y_true) / len(y_true) if len(y_true) > 0 else 0.0
        recall = 1.0 if np.sum(y_true) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    else:
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Confusion matrix elements
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel() if len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1 else (0, 0, 0, 0)
    
    return {
        'Model': model_name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Anomalies_Detected': np.sum(y_pred)
    }

# Compute metrics for all models
metrics_list = [
    safe_metrics(ground_truth, predictions_z, 'Z-Score'),
    safe_metrics(ground_truth, predictions_iso, 'Isolation Forest'),
    safe_metrics(ground_truth, predictions_lstm, 'LSTM Autoencoder')
]

metrics_df = pd.DataFrame(metrics_list)

print('\n' + '='*80)
print('MODEL EVALUATION METRICS')
print('='*80)
print(metrics_df.to_string(index=False))

## 8. Detailed Results Comparison

In [ ]:
print('\n' + '='*80)
print('DETAILED ANALYSIS')
print('='*80)

for idx, row in metrics_df.iterrows():
    print(f'\n{idx+1}. {row["Model"].upper()}')
    print(f'   ─────────────────────────────────')
    print(f'   Precision:  {row["Precision"]:.4f}  (of detected, how many truly anomalous)')
    print(f'   Recall:     {row["Recall"]:.4f}  (of total anomalies, how many detected)')
    print(f'   F1-Score:   {row["F1-Score"]:.4f}  (harmonic mean of precision & recall)')
    print(f'   ─────────────────────────────────')
    print(f'   True Positives:  {int(row["TP"])}   (correctly identified anomalies)')
    print(f'   False Positives: {int(row["FP"])}   (normal flagged as anomaly)')
    print(f'   False Negatives: {int(row["FN"])}   (anomalies missed)')
    print(f'   True Negatives:  {int(row["TN"])}   (correctly identified normal)')
    print(f'   ─────────────────────────────────')
    print(f'   Total detected: {int(row["Anomalies_Detected"])} / {len(ground_truth)}')

# Summary
print(f'\n' + '='*80)
print('SUMMARY')
print('='*80)
best_f1_idx = metrics_df['F1-Score'].idxmax()
best_model = metrics_df.iloc[best_f1_idx]['Model']
best_f1 = metrics_df.iloc[best_f1_idx]['F1-Score']
print(f'\n🏆 Best Model: {best_model} (F1-Score: {best_f1:.4f})')
print(f'\n✓ Average F1-Score: {metrics_df["F1-Score"].mean():.4f}')
print(f'✓ Average Precision: {metrics_df["Precision"].mean():.4f}')
print(f'✓ Average Recall: {metrics_df["Recall"].mean():.4f}')

## 9. Visualizations

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. F1-Score Comparison
ax = axes[0, 0]
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax.bar(metrics_df['Model'], metrics_df['F1-Score'], color=colors, edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('F1-Score', fontweight='bold', fontsize=11)
ax.set_title('F1-Score Comparison', fontweight='bold', fontsize=12)
ax.set_ylim([0, 1])
ax.axhline(0.6, color='red', linestyle='--', linewidth=2, label='Target (0.6)', alpha=0.7)
for bar, val in zip(bars, metrics_df['F1-Score']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, f'{val:.3f}', 
            ha='center', fontweight='bold', fontsize=10)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 2. Precision vs Recall
ax = axes[0, 1]
ax.scatter(metrics_df['Recall'], metrics_df['Precision'], s=300, alpha=0.7, edgecolors='black', linewidth=2)
for idx, row in metrics_df.iterrows():
    ax.annotate(row['Model'], (row['Recall'], row['Precision']), 
                xytext=(5, 5), textcoords='offset points', fontweight='bold', fontsize=9)
ax.set_xlabel('Recall', fontweight='bold', fontsize=11)
ax.set_ylabel('Precision', fontweight='bold', fontsize=11)
ax.set_title('Recall vs Precision', fontweight='bold', fontsize=12)
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)

# 3. Metrics Heatmap
ax = axes[1, 0]
metrics_plot = metrics_df[['Model', 'Precision', 'Recall', 'F1-Score']].set_index('Model')
sns.heatmap(metrics_plot.T, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, cbar=True,
            cbar_kws={'label': 'Score'}, linewidths=2, linecolor='black', 
            annot_kws={'size': 11, 'weight': 'bold'}, vmin=0, vmax=1)
ax.set_title('Metrics Heatmap', fontweight='bold', fontsize=12)
ax.set_ylabel('')

# 4. Confusion Matrix (Best Model)
ax = axes[1, 1]
best_idx = metrics_df['F1-Score'].idxmax()
best_predictions = [predictions_z, predictions_iso, predictions_lstm][best_idx]
cm = confusion_matrix(ground_truth, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 14, 'weight': 'bold'}, linewidths=2, linecolor='black')
ax.set_title(f'Confusion Matrix - {metrics_df.iloc[best_idx]["Model"]}', fontweight='bold', fontsize=12)
ax.set_ylabel('True Label', fontweight='bold', fontsize=11)
ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('Model_Evaluation_Metrics.png', dpi=150, bbox_inches='tight')
print('✓ Saved: Model_Evaluation_Metrics.png')
plt.show()

## 10. Export Results

In [ ]:
# Export metrics to CSV
metrics_df.to_csv('Model_Evaluation_Metrics.csv', index=False)
print('✓ Saved: Model_Evaluation_Metrics.csv')
print(f'\nExport Summary:')
print(metrics_df[['Model', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

## 11. Executive Summary

In [ ]:
print('\n' + '='*80)
print('ANOMALY DETECTION MODELS - EXECUTIVE SUMMARY')
print('='*80)

print('\n📊 DATASET')
print('-'*80)
print(f'Total samples: {len(df_clean):,}')
print(f'Features: {len(df_clean.columns)}')
print(f'Ground truth anomalies: {np.sum(ground_truth):,} ({100*np.sum(ground_truth)/len(ground_truth):.1f}%)')

print('\n🤖 MODELS EVALUATED')
print('-'*80)
for idx, row in metrics_df.iterrows():
    print(f'{idx+1}. {row["Model"]:<20} | F1: {row["F1-Score"]:.4f} | Prec: {row["Precision"]:.4f} | Rec: {row["Recall"]:.4f}')

print('\n🏆 RANKINGS')
print('-'*80)
f1_ranked = metrics_df.sort_values('F1-Score', ascending=False)
for idx, row in f1_ranked.iterrows():
    position = ['🥇 1st', '🥈 2nd', '🥉 3rd'][idx]
    print(f'{position} | {row["Model"]:<20} | F1-Score: {row["F1-Score"]:.4f}')

print('\n💡 INSIGHTS')
print('-'*80)
print(f'✓ Best overall model: {metrics_df.loc[metrics_df["F1-Score"].idxmax(), "Model"]}')
print(f'✓ Highest precision: {metrics_df.loc[metrics_df["Precision"].idxmax(), "Model"]} ({metrics_df["Precision"].max():.4f})')
print(f'✓ Highest recall: {metrics_df.loc[metrics_df["Recall"].idxmax(), "Model"]} ({metrics_df["Recall"].max():.4f})')
print(f'✓ Average F1-Score across all models: {metrics_df["F1-Score"].mean():.4f}')

print('\n' + '='*80)
print('✓ READY FOR DEPLOYMENT')
print('='*80 + '\n')